In [16]:
import warnings
warnings.filterwarnings("ignore")
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from langchain_community.tools import tool
import requests

In [17]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [23]:
# tool creation
@tool
def get_conversion_factor(base_currency:str , target_currency:str) -> float:
    """
    this function fetches the currency conversion factor between a given base_currency and a target currency
    """
    url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'
    response = requests.get(url)
    return response.json()

@tool
def convert(base_currency_value:int , conversion_rate:Annotated[float,InjectedToolArg]) -> float:
    """
    Given a currency conversion rate this function calculates the target currency value from a given base currency value
    """
    return base_currency_value * conversion_rate

In [24]:
# tool binding
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")
llm_with_tools = llm.bind_tools([get_conversion_factor , convert])

message = []

human_message = HumanMessage(
    content = "what is conversion factor between USD and INR , and based on that can you convert 100 USD into INR"
)
message.append(human_message)

ai_message = llm_with_tools.invoke(message)
message.append(ai_message)

In [ ]:

# tool_calling
import json


for tool_call in ai_message.tool_calls:
    if tool_call["name"] == "get_conversion_factor":
        tool_message1 = get_conversion_factor.invoke(tool_call)
        conversion_rate = json.load(
            tool_message1.content
        )["conversion_rate"]
        message.append(tool_message1)
    if tool_call["name"] == "convert":
        tool_call["args"]["conversion_rate"] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        message.append(tool_message2)

In [ ]:
final_result = llm_with_tools.invoke(message)
print(final_result.content)